---
## Dashboard 1: Executive Overview (C-Suite)
**Visuals:** KPI cards, revenue trend line, region bar chart, product donut

**Metrics:** Total Revenue, Profit Margin, YoY Growth, Top Products

In [0]:
%sql
-- [EXEC-1] KPI Cards with YoY Growth
-- Visual: Counter tiles

WITH yearly AS (
  SELECT 
    year, 
    total_revenue, 
    total_profit, 
    profit_margin_pct, 
    total_transactions, 
    active_customers,
    total_units_sold,
    -- Added OVER() to make this a window function instead of an aggregate
    ROUND(total_revenue / total_transactions, 2) AS avg_transaction_value, 
    LAG(total_revenue) OVER (ORDER BY year) AS prev_revenue,
    LAG(total_profit) OVER (ORDER BY year) AS prev_profit
  FROM maven_catalog.gold_schema.kpi_executive_summary
)
SELECT 
  year, 
  ROUND(total_revenue,2), 
  ROUND(total_profit,2), 
  profit_margin_pct, 
  total_transactions, 
  active_customers,
  -- Handling potential division by zero just in case
  ROUND(NULLIF(total_revenue - prev_revenue, 0) / prev_revenue * 100, 2) AS revenue_yoy_pct,
  ROUND(NULLIF(total_profit - prev_profit, 0) / prev_profit * 100, 2) AS profit_yoy_pct
FROM yearly
ORDER BY year DESC

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- [EXEC-2] Revenue Trend Line (Monthly)
-- Visual: Line chart — X: year_month, Y: revenue + profit

SELECT
  year_month,
  ROUND(SUM(total_revenue),2) AS revenue,
  ROUND(SUM(total_profit),2) AS profit,
  SUM(total_units) AS units_sold
FROM maven_catalog.gold_schema.agg_monthly_sales
GROUP BY year_month
ORDER BY year_month

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- [EXEC-3] Revenue by Region (Bar Chart)
-- Visual: Bar — X: sales_region, Y: total_revenue, Color: store_country

SELECT
  sales_region,
  store_country,
  ROUND(SUM(total_revenue),2) AS total_revenue,
  ROUND(SUM(total_profit),2) AS total_profit
FROM maven_catalog.gold_schema.agg_store_performance
GROUP BY sales_region, store_country
ORDER BY total_revenue DESC

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- [EXEC-4] Top 10 Products (Donut Chart)
-- Visual: Donut — Slice: product_brand, Value: revenue

SELECT
  product_brand,
  ROUND(SUM(gross_revenue),2) AS revenue,
  ROUND(SUM(gross_profit),2) AS profit
FROM maven_catalog.gold_schema.agg_product_performance
GROUP BY product_brand
ORDER BY revenue DESC
LIMIT 10

Databricks visualization. Run in Databricks to view.

In [0]:
-- Customer Segment Distribution
SELECT 
  customer_segment,
  COUNT(customer_id) AS customer_count,
  ROUND(SUM(lifetime_revenue), 2) AS total_segment_revenue
FROM maven_catalog.gold_schema.agg_customer_lifetime
GROUP BY customer_segment
ORDER BY customer_count DESC

Databricks visualization. Run in Databricks to view.

In [0]:
-- Top 10 High-Risk Products (Return Rate)
SELECT 
  product_name,
  product_brand,
  units_sold,
  units_returned,
  return_rate
FROM maven_catalog.gold_schema.agg_product_performance
WHERE units_sold > 50 -- Filter out low volume to avoid skewed percentages
ORDER BY return_rate DESC
LIMIT 10

Databricks visualization. Run in Databricks to view.

In [0]:
-- Store Efficiency (Revenue per Sq Ft)
SELECT 
  store_name,
  store_city,
  sales_region,
  ROUND(total_revenue,2),
  total_sqft,
  revenue_per_sqft
FROM maven_catalog.gold_schema.agg_store_performance
ORDER BY revenue_per_sqft DESC

Databricks visualization. Run in Databricks to view.